# GEO-ADAPT Wildfire Summer School 2026
## 00 — Environment Preflight

Run **Run → Run All Cells** in CDSE JupyterLab.

At the end you should see **YOUR COURSE ENVIRONMENT IS READY**.  
If you see any **FAIL** items, send the error text or a screenshot to the trainers before the course.

`WARN` means an optional component is unavailable and is not necessarily a blocker.


## 1. Configuration

If you use your own Google Cloud project enabled for Earth Engine, replace the project ID below.


In [ ]:
GEE_PROJECT_ID = "ee-andreydara"

# Optional: NASA FIRMS API key.
# Prefer setting this as an environment variable:
# FIRMS_MAP_KEY = "..."


In [ ]:
from __future__ import annotations

import importlib
import os
import platform
import sys
from pathlib import Path

RESULTS = []

def record(name, status, detail=""):
    status = status.upper()
    RESULTS.append((name, status, str(detail)))
    icon = {"PASS": "✓", "WARN": "!", "FAIL": "✗"}.get(status, "?")
    print(f"{icon} {status:<4} | {name}" + (f" — {detail}" if detail else ""))

def safe_test(name, fn, optional=False):
    try:
        detail = fn()
        record(name, "PASS", detail or "")
        return True
    except Exception as e:
        record(name, "WARN" if optional else "FAIL",
               f"{type(e).__name__}: {e}")
        return False

def import_test(import_name, display_name=None, optional=False):
    display_name = display_name or import_name
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "version unavailable")
        record(display_name, "PASS", version)
        return module
    except Exception as e:
        record(display_name, "WARN" if optional else "FAIL",
               f"{type(e).__name__}: {e}")
        return None

print("Preflight helpers loaded.")


## 2. Python and package checks

In [ ]:
record("Python", "PASS", sys.version.split()[0])
record("Operating system", "PASS", f"{platform.system()} {platform.release()}")

np = import_test("numpy", "NumPy")
pd = import_test("pandas", "Pandas")
gpd = import_test("geopandas", "GeoPandas")
rasterio = import_test("rasterio", "Rasterio")
rioxarray = import_test("rioxarray", "rioxarray")
xr = import_test("xarray", "xarray")
mpl = import_test("matplotlib", "Matplotlib")
sklearn = import_test("sklearn", "scikit-learn")
requests = import_test("requests", "Requests")
pystac_client = import_test("pystac_client", "pystac-client")
openeo = import_test("openeo", "openEO client")
ee = import_test("ee", "Earth Engine Python API")
geemap = import_test("geemap", "geemap", optional=True)


### Optional package install

Only use this if a trainer asks you to:

```python
%pip install -q earthengine-api geemap pystac-client openeo
```


## 3. Basic geospatial functionality

In [ ]:
def test_dataframe():
    import pandas as pd
    df = pd.DataFrame({"site": ["A", "B"], "value": [1, 2]})
    assert int(df["value"].sum()) == 3
    return "DataFrame operations work"

safe_test("Pandas calculation", test_dataframe)


In [ ]:
def test_geodataframe():
    import geopandas as gpd
    from shapely.geometry import Point
    gdf = gpd.GeoDataFrame(
        {"site": ["Ohrid"]},
        geometry=[Point(20.8016, 41.1231)],
        crs="EPSG:4326",
    )
    assert len(gdf) == 1
    assert str(gdf.crs) == "EPSG:4326"
    return "GeoDataFrame + CRS work"

safe_test("GeoPandas / Shapely", test_geodataframe)


In [ ]:
def test_matplotlib():
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 2.4))
    ax.plot([0, 1, 2], [0, 1, 0])
    ax.set_title("Preflight plot")
    plt.show()
    return "Plot rendered"

safe_test("Matplotlib rendering", test_matplotlib)


## 4. Persistent CDSE storage

In [ ]:
def test_mystorage():
    target = Path.home() / "mystorage"
    if not target.exists():
        raise FileNotFoundError(
            f"{target} does not exist. Are you running this in CDSE JupyterLab?"
        )
    probe = target / "_geo_adapt_preflight_test.txt"
    probe.write_text("GEO-ADAPT preflight OK\n", encoding="utf-8")
    text = probe.read_text(encoding="utf-8").strip()
    probe.unlink()
    assert text == "GEO-ADAPT preflight OK"
    return f"Read/write/delete works in {target}"

safe_test("Persistent mystorage", test_mystorage)


## 5. Internet connectivity

In [ ]:
def test_internet():
    import requests
    r = requests.get("https://dataspace.copernicus.eu", timeout=15)
    r.raise_for_status()
    return f"HTTP {r.status_code}"

safe_test("Internet / CDSE website", test_internet)


## 6. CDSE openEO backend

In [ ]:
def test_openeo_backend():
    import openeo
    con = openeo.connect("openeo.dataspace.copernicus.eu")
    caps = con.capabilities()
    version = getattr(caps, "api_version", None)
    return "Backend reachable" + (f"; API {version}" if version else "")

safe_test("CDSE openEO backend", test_openeo_backend)


## 7. Public STAC access

In [ ]:
def test_stac():
    from pystac_client import Client
    catalog = Client.open("https://earth-search.aws.element84.com/v1")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=[20.6, 40.9, 21.1, 41.4],
        datetime="2024-07-01/2024-07-10",
        max_items=1,
    )
    items = list(search.items())
    if not items:
        raise RuntimeError("STAC endpoint responded but returned no test item.")
    return f"STAC reachable; example item: {items[0].id}"

safe_test("Public STAC catalogue", test_stac)


## 8. Google Earth Engine

The test tries existing credentials first. If they are missing/invalid, it starts the normal Earth Engine authentication flow.


In [ ]:
def test_gee():
    import ee

    try:
        ee.Initialize(project=GEE_PROJECT_ID)
    except Exception:
        print("Existing Earth Engine credentials were not usable.")
        print("Starting Earth Engine authentication...")
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT_ID)

    value = ee.Number(10).pow(2).multiply(3).subtract(5).getInfo()
    if value != 295:
        raise RuntimeError(f"Unexpected Earth Engine result: {value}")

    bands = ee.Image("USGS/SRTMGL1_003").bandNames().getInfo()
    if "elevation" not in bands:
        raise RuntimeError(f"Unexpected SRTM bands: {bands}")

    return f"Project '{GEE_PROJECT_ID}'; compute + SRTM catalogue access OK"

safe_test("Google Earth Engine", test_gee)


## 9. Optional NASA FIRMS API

This is **not a blocker** if you have not configured your FIRMS key yet.


In [ ]:
def test_firms():
    import os
    import requests

    key = globals().get("FIRMS_MAP_KEY") or os.environ.get("FIRMS_MAP_KEY")
    if not key:
        raise RuntimeError("No FIRMS_MAP_KEY configured — live test skipped.")

    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{key}/VIIRS_SNPP_NRT/20.5,40.8,21.2,41.5/1"
    )
    r = requests.get(url, timeout=20)
    r.raise_for_status()

    if "latitude" not in r.text.lower():
        raise RuntimeError("FIRMS response did not look like expected CSV.")
    return "FIRMS API reachable"

safe_test("NASA FIRMS API", test_firms, optional=True)


## 10. Final readiness summary

In [ ]:
from collections import Counter

counts = Counter(status for _, status, _ in RESULTS)

print("\n" + "=" * 68)
print("PREFLIGHT SUMMARY")
print("=" * 68)
print(f"PASS: {counts.get('PASS', 0)}")
print(f"WARN: {counts.get('WARN', 0)}")
print(f"FAIL: {counts.get('FAIL', 0)}")

fails = [(name, detail) for name, status, detail in RESULTS if status == "FAIL"]
warns = [(name, detail) for name, status, detail in RESULTS if status == "WARN"]

if fails:
    print("\nBLOCKERS:")
    for name, detail in fails:
        print(f"  ✗ {name}: {detail}")
    print("\n" + "!" * 68)
    print("YOUR COURSE ENVIRONMENT IS NOT READY YET")
    print("Please send the FAIL messages above to the trainers.")
    print("!" * 68)
else:
    print("\n" + "=" * 68)
    print("YOUR COURSE ENVIRONMENT IS READY")
    print("=" * 68)
    if warns:
        print("\nOptional warnings:")
        for name, detail in warns:
            print(f"  ! {name}: {detail}")


### If something fails

Send the trainers:
- a screenshot of the **PREFLIGHT SUMMARY**;
- the full text of the failed check;
- confirmation that you are running this in the **CDSE JupyterLab Geo Science environment**.
